# Problem 1: Heterodimer Protein Structure Analysis

## Problem Summary
A structural laboratory has found an incomplete model of a PROBLEM protein that works as a heterodimer complex. We need to analyze its function, structure quality, and fix any issues.

## Biological Context

**What is a heterodimer?**
A heterodimer is a protein complex formed by two DIFFERENT protein chains (as opposed to a homodimer which has two identical chains). Heterodimers are common in:
- Transcription factors (e.g., Myc-Max, Jun-Fos)
- Receptor signaling (e.g., integrins)
- Enzyme regulation

**Why CA-only coordinates?**
The problem provides only C-alpha (CA) coordinates, which is a simplified representation:
- CA atoms represent the protein backbone trace
- Full backbone requires N, CA, C, O atoms
- Side chains add specificity but CA captures overall fold
- DSSP requires full backbone, so we need reconstruction

## Points Distribution
- a) Function identification: 0.5 pts
- b) PFAM family identification: 0.5 pts  
- c) SCOP fold classification: 0.5 pts
- d) DSSP secondary structure: 0.5 pts
- e) Secondary structure prediction comparison: 0.5 pts
- f) Structure validation with ProSA: 0.5 pts
- g) Identify structural problems: 0.5 pts
- h) Dimer formation assessment: 0.5 pts
- i) Functional residue conservation: 0.5 pts
- j) Active site visualization: 0.5 pts
- k) Fix structural problems: 0.5 pts
- l) New model dimer assessment: 0.5 pts

**Total: 6 points**

---
## Configuration

In [1]:
# ============================================================
# CONFIGURATION - Modify these paths as needed
# ============================================================

# Input file
PROBLEM_FILE = "Exam/problem_1.txt"

# Output directory and file naming prefix
OUTPUT_DIR = "Problem_1_outputs"
OUTPUT_PREFIX = "p38"  # Files will be named p38b1.hmm, p38d1.dssp, etc.

# Database paths (local databases)
SWISSPROT_DB = "databases/swissprot/swissprot"
PDBAA_DB = "databases/pdb_seq/pdbaa"
PDBAA_FASTA = "databases/pdb_seq/pdbaa.fasta"
PFAM_DB = "databases/hmm/Pfam/Pfam-A.hmm"

# Working directories
TEMP_DIR = "temp"
TEMPLATES_DIR = "Templates"
ALIGNMENTS_DIR = "Alignments"
MODELLER_DIR = "Modeller_Templates"

In [2]:
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to path
sys.path.insert(0, str(Path.cwd()))

# Import project modules
from src.Homology.retrieve import TemplateRetriever
from src.Homology.domains import TemplateProcessor
from src.Analysis.assessment import (
    identify_protein_family, 
    create_hmm_profile, 
    run_dssp_analysis,
    identify_functional_residues,
    validate_model_regions,
    fix_model_problems,
    extract_sequence_from_pdb,
    ModelAssessor
)
from src.Analysis.visualization import (
    visualize_active_site,
    visualize_problematic_regions,
    compare_structures,
    StructureVisualizer
)
from src.modeller.scripts import (
    generate_single_template_script,
    generate_loop_refinement_script,
    ModellerRunner
)
from src.UI.app import run_dssp, DomainVisualizer

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.PDB import PDBParser, PDBIO, Select

# Create output directories
for d in [OUTPUT_DIR, TEMP_DIR, TEMPLATES_DIR, ALIGNMENTS_DIR, MODELLER_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Output prefix: {OUTPUT_PREFIX}")

Output directory: Problem_1_outputs
Output prefix: p38


---
## Step 0: Data Preparation

### Biological Background: PDB File Format
PDB files contain atomic coordinates with specific fields:
- **Columns 1-6**: Record type (ATOM, HETATM)
- **Columns 13-16**: Atom name (CA = C-alpha)
- **Columns 18-20**: Residue name (3-letter code)
- **Column 22**: Chain identifier (A, B, etc.)
- **Columns 23-26**: Residue number
- **Columns 31-54**: X, Y, Z coordinates
- **Columns 61-66**: B-factor (temperature factor, indicates flexibility)

In [3]:
# Three-letter to one-letter amino acid mapping
THREE_TO_ONE = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

def parse_problem_file(filepath):
    """Parse the problem file to extract CA coordinates and sequences."""
    chain_a_atoms = []
    chain_b_atoms = []
    
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('ATOM'):
                # Parse PDB ATOM record
                atom_name = line[12:16].strip()
                resname = line[17:20].strip()
                chain = line[21]
                resnum = int(line[22:26])
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                bfactor = float(line[60:66])
                
                atom_data = {
                    'atom': atom_name,
                    'resname': resname,
                    'chain': chain,
                    'resnum': resnum,
                    'x': x, 'y': y, 'z': z,
                    'bfactor': bfactor,
                    'aa': THREE_TO_ONE.get(resname, 'X')
                }
                
                if chain == 'A':
                    chain_a_atoms.append(atom_data)
                elif chain == 'B':
                    chain_b_atoms.append(atom_data)
    
    seq_a = ''.join([a['aa'] for a in chain_a_atoms])
    seq_b = ''.join([a['aa'] for a in chain_b_atoms])
    
    return chain_a_atoms, chain_b_atoms, seq_a, seq_b

# Parse the problem file
chain_a_atoms, chain_b_atoms, seq_a, seq_b = parse_problem_file(PROBLEM_FILE)

print(f"Chain A: {len(seq_a)} residues")
print(f"Sequence: {seq_a}")
print(f"\nChain B: {len(seq_b)} residues")
print(f"Sequence: {seq_b}")

Chain A: 89 residues
Sequence: NVKRRTHNVLERQRRNELKRSFFAFRDQIPELENNEKAPKVVILKKFTATATATTAYILSVQAEEQKLISEEDLLRKRREQLKHKLEQL

Chain B: 83 residues
Sequence: DALAHHNALELDLADHIKDSFHSLSLFLDSVPSLQGEKASRAQILDKFTEYIQYMRRKNHTHQQDIDDLKRQNALLEQQVRAL


In [4]:
def write_ca_pdb(atoms, output_path):
    """Write CA atoms to PDB file."""
    with open(output_path, 'w') as f:
        for i, atom in enumerate(atoms, 1):
            f.write(f"ATOM  {i:5d}  CA  {atom['resname']:3s} {atom['chain']}{atom['resnum']:4d}    "
                    f"{atom['x']:8.3f}{atom['y']:8.3f}{atom['z']:8.3f}  1.00{atom['bfactor']:6.2f}           C\n")
        f.write("END\n")

def write_fasta(sequence, seq_id, output_path):
    """Write sequence to FASTA file."""
    record = SeqRecord(Seq(sequence), id=seq_id, description="")
    SeqIO.write([record], output_path, "fasta")

# Save sequences as FASTA
fasta_a = Path(OUTPUT_DIR) / "chain_a.fa"
fasta_b = Path(OUTPUT_DIR) / "chain_b.fa"
write_fasta(seq_a, "ChainA", fasta_a)
write_fasta(seq_b, "ChainB", fasta_b)

# Save CA structures
pdb_a = Path(OUTPUT_DIR) / "chain_a_ca.pdb"
pdb_b = Path(OUTPUT_DIR) / "chain_b_ca.pdb"
write_ca_pdb(chain_a_atoms, pdb_a)
write_ca_pdb(chain_b_atoms, pdb_b)

# Combined structure
pdb_combined = Path(OUTPUT_DIR) / "heterodimer_ca.pdb"
write_ca_pdb(chain_a_atoms + chain_b_atoms, pdb_combined)

print(f"Saved FASTA files: {fasta_a}, {fasta_b}")
print(f"Saved PDB files: {pdb_a}, {pdb_b}, {pdb_combined}")

Saved FASTA files: Problem_1_outputs/chain_a.fa, Problem_1_outputs/chain_b.fa
Saved PDB files: Problem_1_outputs/chain_a_ca.pdb, Problem_1_outputs/chain_b_ca.pdb, Problem_1_outputs/heterodimer_ca.pdb


---
## a) Function Identification (0.5 pts)

### Biological Background: How to Identify Protein Function

**Strategy:**
1. **Sequence similarity search** (BLAST) - Proteins with similar sequences usually have similar functions
2. Look at the **top hits** - Pay attention to:
   - Protein names (e.g., "Myc proto-oncogene protein")
   - Organism (human, mouse, etc.)
   - Description/annotations

**Interpreting BLAST Results:**
- **E-value < 1e-10**: Very significant hit, likely same function
- **Identity > 30%**: Likely same fold and similar function
- **Coverage > 80%**: Hit covers most of your sequence

**What to look for in the answer:**
- Protein name from SwissProt annotation
- Biological function (e.g., transcription factor, enzyme)
- How the heterodimer relates to function

In [5]:
def blast_sequence(fasta_file, database, output_file, outfmt="6 sacc bitscore evalue pident qcovs stitle"):
    """Run BLAST search against local database."""
    cmd = [
        "psiblast",
        "-query", str(fasta_file),
        "-db", database,
        "-out", str(output_file),
        "-outfmt", outfmt,
        "-num_iterations", "1",
        "-evalue", "0.001"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"BLAST error: {result.stderr}")
    return Path(output_file)

# BLAST Chain A against SwissProt
blast_a_sp = Path(OUTPUT_DIR) / "blast_chainA_swissprot.out"
blast_sequence(fasta_a, SWISSPROT_DB, blast_a_sp)

# BLAST Chain B against SwissProt
blast_b_sp = Path(OUTPUT_DIR) / "blast_chainB_swissprot.out"
blast_sequence(fasta_b, SWISSPROT_DB, blast_b_sp)

print("BLAST search completed. Check results:")
print(f"  Chain A: {blast_a_sp}")
print(f"  Chain B: {blast_b_sp}")

BLAST search completed. Check results:
  Chain A: Problem_1_outputs/blast_chainA_swissprot.out
  Chain B: Problem_1_outputs/blast_chainB_swissprot.out


In [6]:
# Display BLAST results
def display_blast_results(filepath):
    """Display BLAST results from tabular output."""
    if filepath.exists() and filepath.stat().st_size > 0:
        df = pd.read_csv(filepath, sep='\t', header=None,
                        names=['Subject', 'BitScore', 'E-value', 'Identity', 'Coverage', 'Title'])
        return df.head(10)
    return pd.DataFrame()

print("\n=== Chain A BLAST Results ===")
blast_a_df = display_blast_results(blast_a_sp)
display(blast_a_df)

print("\n=== Chain B BLAST Results ===")
blast_b_df = display_blast_results(blast_b_sp)
display(blast_b_df)


=== Chain A BLAST Results ===


,Subject,BitScore,E-value,Identity,Coverage,Title
0,P01106,155.0,4.360000e-46,89.888,100,RecName: Full=Myc proto-oncogene protein; AltN...
1,Q9MZU0,151.0,1.790000e-44,87.640,100,RecName: Full=Myc proto-oncogene protein; AltN...
2,Q9MZT9,151.0,1.880000e-44,87.640,100,RecName: Full=Myc proto-oncogene protein; AltN...
3,A2T7L5,149.0,6.020000e-44,86.517,100,RecName: Full=Myc proto-oncogene protein; AltN...
4,Q28350,149.0,6.560000e-44,86.517,100,RecName: Full=Myc proto-oncogene protein; AltN...
5,Q9MZT8,148.0,1.550000e-43,85.393,100,RecName: Full=Myc proto-oncogene protein; AltN...
6,Q9MZT6,148.0,1.620000e-43,85.393,100,RecName: Full=Myc proto-oncogene protein; AltN...
7,P49032,148.0,1.630000e-43,86.517,100,RecName: Full=Myc proto-oncogene protein; AltN...
8,P49033,147.0,4.980000e-43,85.393,100,RecName: Full=Myc proto-oncogene protein; AltN...
9,P22555,147.0,7.130000e-43,85.393,100,RecName: Full=Myc proto-oncogene protein; AltN...



=== Chain B BLAST Results ===


,Subject,BitScore,E-value,Identity,Coverage,Title
0,P28574,140.0,3.660000e-43,86.747,100,RecName: Full=Protein max; AltName: Full=Myc-a...
1,P52164,140.0,3.780000e-43,86.747,100,RecName: Full=Protein max; AltName: Full=Myc-a...
2,P61244,140.0,3.820000e-43,86.747,100,RecName: Full=Protein max; AltName: Full=Class...
3,P52162,139.0,4.360000e-43,86.747,100,RecName: Full=Protein max; AltName: Full=Myc-a...
4,Q07016,133.0,1.670000e-40,85.000,96,RecName: Full=Protein max; Short=xMAX; AltName...
5,P52161,132.0,5.710000e-40,80.682,100,RecName: Full=Protein max; AltName: Full=Myc-a...
6,P91664,89.7,2.150000e-23,57.500,96,RecName: Full=Protein max; Short=dMax; AltName...
7,Q18711,59.3,4.620000e-11,45.205,88,RecName: Full=Protein mxl-3; AltName: Full=Bas...
8,O08789,42.4,6.680000e-05,26.923,94,RecName: Full=Max-binding protein MNT; AltName...
9,G5EEH5,41.2,8.830000e-05,37.037,95,RecName: Full=Max-like protein 1 [Caenorhabdit...


In [7]:
# ANSWER TEMPLATE - Fill in based on BLAST results above

# Extract protein names from top hits
chain_a_protein = "" if len(blast_a_df) == 0 else blast_a_df.iloc[0]['Title'].split('[')[0].strip()
chain_b_protein = "" if len(blast_b_df) == 0 else blast_b_df.iloc[0]['Title'].split('[')[0].strip()

function_answer = f"""
=== ANSWER a) Function Identification ===

Based on BLAST analysis against SwissProt:

CHAIN A:
- Best hit: {blast_a_df.iloc[0]['Subject'] if len(blast_a_df) > 0 else 'N/A'}
- Protein: {chain_a_protein}
- Identity: {blast_a_df.iloc[0]['Identity'] if len(blast_a_df) > 0 else 'N/A'}%
- E-value: {blast_a_df.iloc[0]['E-value'] if len(blast_a_df) > 0 else 'N/A'}

CHAIN B:
- Best hit: {blast_b_df.iloc[0]['Subject'] if len(blast_b_df) > 0 else 'N/A'}
- Protein: {chain_b_protein}
- Identity: {blast_b_df.iloc[0]['Identity'] if len(blast_b_df) > 0 else 'N/A'}%
- E-value: {blast_b_df.iloc[0]['E-value'] if len(blast_b_df) > 0 else 'N/A'}

HETERODIMER FUNCTION:
Based on the BLAST results, this heterodimer appears to be:
[FILL IN: e.g., "The Myc-Max transcription factor complex that regulates cell proliferation genes"]

Biological significance:
[FILL IN: e.g., "Myc is an oncogene that requires Max for DNA binding. The heterodimer binds E-box sequences."]
"""

print(function_answer)


=== ANSWER a) Function Identification ===

Based on BLAST analysis against SwissProt:

CHAIN A:
- Best hit: P01106
- Protein: RecName: Full=Myc proto-oncogene protein; AltName: Full=Class E basic helix-loop-helix protein 39; Short=bHLHe39; AltName: Full=Proto-oncogene c-Myc; AltName: Full=Transcription factor p64
- Identity: 89.888%
- E-value: 4.36e-46

CHAIN B:
- Best hit: P28574
- Protein: RecName: Full=Protein max; AltName: Full=Myc-associated factor X; AltName: Full=Myc-binding novel HLH/LZ protein; AltName: Full=Protein myn
- Identity: 86.747%
- E-value: 3.6600000000000005e-43

HETERODIMER FUNCTION:
Based on the BLAST results, this heterodimer appears to be:
[FILL IN: e.g., "The Myc-Max transcription factor complex that regulates cell proliferation genes"]

Biological significance:
[FILL IN: e.g., "Myc is an oncogene that requires Max for DNA binding. The heterodimer binds E-box sequences."]



In [8]:
import requests

def get_uniprot_function(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    
    # Extract function from comments
    functions = []
    for comment in data.get("comments", []):
        if comment.get("commentType") == "FUNCTION":
            for text in comment.get("texts", []):
                functions.append(text["value"])
    
    return functions

# Your hits
chains = {"A": "P01106", "B": "P28574"}

for chain, accession in chains.items():
    funcs = get_uniprot_function(accession)
    print(f"\nChain {chain} ({accession}):")
    for f in funcs:
        print(f"  {f}")


Chain A (P01106):
  Transcription factor that binds DNA in a non-specific manner, yet also specifically recognizes the core sequence 5'-CAC[GA]TG-3' (PubMed:24940000, PubMed:25956029). Activates the transcription of growth-related genes (PubMed:24940000, PubMed:25956029). Binds to the VEGFA promoter, promoting VEGFA production and subsequent sprouting angiogenesis (PubMed:24940000, PubMed:25956029). Regulator of somatic reprogramming, controls self-renewal of embryonic stem cells (By similarity). Functions with TAF6L to activate target gene expression through RNA polymerase II pause release (By similarity). Positively regulates transcription of HNRNPA1, HNRNPA2 and PTBP1 which in turn regulate splicing of pyruvate kinase PKM by binding repressively to sequences flanking PKM exon 9, inhibiting exon 9 inclusion and resulting in exon 10 inclusion and production of the PKM M2 isoform (PubMed:20010808)

Chain B (P28574):
  Transcription regulator. Forms a sequence-specific DNA-binding prot

---
## b) PFAM Family Identification (0.5 pts)

### Biological Background: Protein Domains and Pfam

**What is Pfam?**
- A database of protein domain families
- Each family has a Hidden Markov Model (HMM) profile
- HMMs capture the conserved patterns in a protein family

**Why domains matter:**
- Proteins are often composed of multiple domains
- Domains are functional/structural units that can appear in different proteins
- Example: The HLH (Helix-Loop-Helix) domain appears in many transcription factors

**How hmmscan works:**
1. Takes your sequence as query
2. Searches against all Pfam HMM profiles
3. Reports which domains are found and where

**Output files needed:**
- `p38b1.hmm` - HMM profile for protein 1's family
- `p38b2.hmm` - HMM profile for protein 2's family

In [9]:
def search_pfam(fasta_file, output_file):
    """Search sequence against local Pfam database using hmmscan."""
    cmd = [
        "hmmscan",
        "--tblout", str(output_file),
        "-E", "1e-5",
        PFAM_DB,
        str(fasta_file)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"hmmscan error: {result.stderr}")
    return Path(output_file)

def parse_hmmscan_tblout(filepath):
    """Parse hmmscan tblout format."""
    results = []
    if not Path(filepath).exists():
        return pd.DataFrame()
    with open(filepath) as f:
        for line in f:
            if not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 6:
                    results.append({
                        'Family': parts[0],
                        'Accession': parts[1],
                        'Query': parts[2],
                        'E-value': float(parts[4]),
                        'Score': float(parts[5]),
                        'Description': ' '.join(parts[18:]) if len(parts) > 18 else ''
                    })
    return pd.DataFrame(results)

# Search Pfam for both chains
pfam_a = Path(OUTPUT_DIR) / "pfam_chainA.out"
pfam_b = Path(OUTPUT_DIR) / "pfam_chainB.out"

print("Running Pfam domain search...")
search_pfam(fasta_a, pfam_a)
search_pfam(fasta_b, pfam_b)

print(f"Pfam results saved to: {pfam_a}, {pfam_b}")

# Parse the results into DataFrames
print("\n=== Chain A Pfam Domains ===")
pfam_a_df = parse_hmmscan_tblout(pfam_a)
if len(pfam_a_df) > 0:
    display(pfam_a_df)
else:
    print("No Pfam domains found for Chain A")

print("\n=== Chain B Pfam Domains ===")
pfam_b_df = parse_hmmscan_tblout(pfam_b)
if len(pfam_b_df) > 0:
    display(pfam_b_df)
else:
    print("No Pfam domains found for Chain B")

Running Pfam domain search...
Pfam results saved to: Problem_1_outputs/pfam_chainA.out, Problem_1_outputs/pfam_chainB.out

=== Chain A Pfam Domains ===


,Family,Accession,Query,E-value,Score,Description
0,Myc-LZ,PF02344.21,ChainA,1.700000e-12,47.4,Myc leucine zipper domain
1,HLH,PF00010.32,ChainA,5.900000e-11,42.6,Helix-loop-helix DNA-binding domain



=== Chain B Pfam Domains ===


,Family,Accession,Query,E-value,Score,Description
0,HLH,PF00010.32,ChainB,4.100000e-07,30.3,Helix-loop-helix DNA-binding domain


In [10]:
def extract_hmm_from_pfam(family_name, pfam_accession, output_file):
    """
    Extract specific HMM profile from Pfam database.
    
    Note: hmmfetch uses the FAMILY NAME (e.g., 'Myc-LZ') as the key,
    NOT the accession number (e.g., 'PF02344').
    """
    # Try with family name first (this is how Pfam HMMs are indexed)
    cmd = ["hmmfetch", "-o", str(output_file), PFAM_DB, family_name]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and Path(output_file).exists() and Path(output_file).stat().st_size > 0:
        print(f"[SUCCESS] Saved HMM profile: {output_file}")
        return True
    
    # Try with accession (some databases are indexed by accession)
    acc_clean = pfam_accession.split('.')[0]
    cmd = ["hmmfetch", "-o", str(output_file), PFAM_DB, acc_clean]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and Path(output_file).exists() and Path(output_file).stat().st_size > 0:
        print(f"[SUCCESS] Saved HMM profile: {output_file}")
        return True
    
    # If both fail, provide manual download instructions
    print(f"[WARNING] Could not extract HMM for {family_name} ({acc_clean})")
    print(f"\n=== MANUAL DOWNLOAD INSTRUCTIONS ===")
    print(f"1. Go to: https://www.ebi.ac.uk/interpro/entry/pfam/{acc_clean}")
    print(f"2. Click 'Curation' tab -> 'Download' -> 'HMM'")
    print(f"3. Save as: {output_file}")
    print(f"\nAlternatively, use wget:")
    print(f"  wget -O {output_file} 'https://www.ebi.ac.uk/interpro/wwwapi//entry/pfam/{acc_clean}?annotation=hmm'")
    return False

# Extract HMM profiles for identified families
hmm_1_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}b1.hmm"
hmm_2_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}b2.hmm"

print("=== Extracting HMM Profiles ===\n")

if len(pfam_a_df) > 0:
    family_a = pfam_a_df.iloc[0]['Family']
    pfam_acc_a = pfam_a_df.iloc[0]['Accession']
    print(f"Chain A: {family_a} ({pfam_acc_a})")
    extract_hmm_from_pfam(family_a, pfam_acc_a, hmm_1_output)
    print()

if len(pfam_b_df) > 0:
    family_b = pfam_b_df.iloc[0]['Family']
    pfam_acc_b = pfam_b_df.iloc[0]['Accession']
    print(f"Chain B: {family_b} ({pfam_acc_b})")
    extract_hmm_from_pfam(family_b, pfam_acc_b, hmm_2_output)

=== Extracting HMM Profiles ===

Chain A: Myc-LZ (PF02344.21)
[SUCCESS] Saved HMM profile: Problem_1_outputs/p38b1.hmm

Chain B: HLH (PF00010.32)
[SUCCESS] Saved HMM profile: Problem_1_outputs/p38b2.hmm


---
## c) SCOP Fold Classification (0.5 pts)

### Biological Background: Protein Fold Classification

**What is SCOP?**
SCOP (Structural Classification of Proteins) organizes proteins hierarchically:
1. **Class**: All-alpha, all-beta, alpha+beta, alpha/beta
2. **Fold**: Same major secondary structures in same arrangement
3. **Superfamily**: Probable common evolutionary origin
4. **Family**: Clear evolutionary relationship

**How to find SCOP classification:**
1. Use the PDB IDs from your BLAST results
2. Search SCOPe: https://scop.berkeley.edu/
3. Or use InterPro which links to SCOP

**Alternative: CATH database**
- Similar hierarchical classification
- https://www.cathdb.info/

In [11]:
# BLAST against PDB to find structural homologs
blast_a_pdb = Path(OUTPUT_DIR) / "blast_chainA_pdb.out"
blast_b_pdb = Path(OUTPUT_DIR) / "blast_chainB_pdb.out"

blast_sequence(fasta_a, PDBAA_DB, blast_a_pdb)
blast_sequence(fasta_b, PDBAA_DB, blast_b_pdb)

print("\n=== Chain A PDB Homologs ===")
pdb_a_df = display_blast_results(blast_a_pdb)
display(pdb_a_df.head(5))

print("\n=== Chain B PDB Homologs ===")
pdb_b_df = display_blast_results(blast_b_pdb)
display(pdb_b_df.head(5))


=== Chain A PDB Homologs ===


,Subject,BitScore,E-value,Identity,Coverage,Title
0,1NKP_A,150.0,7.090000e-49,89.888,100,"Chain A, Myc proto-oncogene protein [Homo sapi..."
1,6G6J_A,150.0,1.060000e-48,89.888,100,"Chain A, Myc proto-oncogene protein [Homo sapi..."
2,5I4Z_A,142.0,5.550000e-45,85.393,100,"Chain A, Myc proto-oncogene protein [Homo sapi..."
3,8OTT_M,97.4,3.770000e-28,85.000,67,"Chain M, Myc proto-oncogene protein [Homo sapi..."
4,1A93_A,56.6,3.630000e-12,96.667,34,"Chain A, MYC PROTO-ONCOGENE PROTEIN [Homo sapi..."



=== Chain B PDB Homologs ===


,Subject,BitScore,E-value,Identity,Coverage,Title
0,1AN2_A,139.0,3.050000e-44,86.747,100,"Chain A, PROTEIN (TRANSCRIPTION FACTOR MAX (TF..."
1,5EYO_A,138.0,4.350000e-44,86.747,100,"Chain A, Protein max [Homo sapiens]"
2,6G6J_B,138.0,4.830000e-44,86.747,100,"Chain B, Protein max [Homo sapiens]"
3,1NKP_B,138.0,5.330000e-44,86.747,100,"Chain B, Max protein [Homo sapiens]"
4,8OTS_N,138.0,5.410000e-44,86.747,100,"Chain N, Protein max [Homo sapiens]"


In [12]:
# Extract PDB IDs for SCOP search
pdb_ids_a = [row['Subject'].split('_')[0] for _, row in pdb_a_df.head(3).iterrows()] if len(pdb_a_df) > 0 else []
pdb_ids_b = [row['Subject'].split('_')[0] for _, row in pdb_b_df.head(3).iterrows()] if len(pdb_b_df) > 0 else []

scop_answer = f"""
=== ANSWER c) SCOP Fold Classification ===

To determine SCOP fold classification:

1. Go to SCOPe database: https://scop.berkeley.edu/

2. Search for these PDB IDs from BLAST results:
   Chain A homologs: {pdb_ids_a}
   Chain B homologs: {pdb_ids_b}

3. Look for the SCOP classification in the results:
   - Class (e.g., "All alpha proteins")
   - Fold (e.g., "DNA/RNA-binding 3-helical bundle")
   - Superfamily (e.g., "Homeodomain-like")
   - Family (e.g., "Myc-type, bHLH domain")

CHAIN A SCOP FOLD:
[FILL IN after searching SCOPe]

CHAIN B SCOP FOLD:
[FILL IN after searching SCOPe]

Note: Both chains likely share the same fold if they form a heterodimer
with a symmetric interface (e.g., both have HLH domains).
"""

print(scop_answer)


=== ANSWER c) SCOP Fold Classification ===

To determine SCOP fold classification:

1. Go to SCOPe database: https://scop.berkeley.edu/

2. Search for these PDB IDs from BLAST results:
   Chain A homologs: ['1NKP', '6G6J', '5I4Z']
   Chain B homologs: ['1AN2', '5EYO', '6G6J']

3. Look for the SCOP classification in the results:
   - Class (e.g., "All alpha proteins")
   - Fold (e.g., "DNA/RNA-binding 3-helical bundle")
   - Superfamily (e.g., "Homeodomain-like")
   - Family (e.g., "Myc-type, bHLH domain")

CHAIN A SCOP FOLD:
[FILL IN after searching SCOPe]

CHAIN B SCOP FOLD:
[FILL IN after searching SCOPe]

Note: Both chains likely share the same fold if they form a heterodimer
with a symmetric interface (e.g., both have HLH domains).



---
## d) DSSP Secondary Structure (0.5 pts)

### Biological Background: DSSP Algorithm

**What is DSSP?**
DSSP (Dictionary of Secondary Structure of Proteins) assigns secondary structure based on:
- Hydrogen bonding patterns in the backbone
- Requires full backbone atoms (N, CA, C, O)

**DSSP codes:**
- H = Alpha helix
- B = Beta bridge
- E = Extended strand (beta sheet)
- G = 3-10 helix
- I = Pi helix
- T = Turn
- S = Bend
- " " = Coil/loop

**Problem: We only have CA atoms!**
Solution: Model full structure using MODELLER

### MODELLER Bash Commands

In [14]:
# Find best template from PDB
template_a = pdb_a_df.iloc[0]['Subject'] if len(pdb_a_df) > 0 else None
template_b = pdb_b_df.iloc[0]['Subject'] if len(pdb_b_df) > 0 else None

print(f"Best template for Chain A: {template_a}")
print(f"Best template for Chain B: {template_b}")

# Download templates
def download_pdb(pdb_id, output_dir=TEMPLATES_DIR):
    """Download PDB file from RCSB."""
    os.makedirs(output_dir, exist_ok=True)
    pdb_code = pdb_id.split('_')[0].lower() if '_' in pdb_id else pdb_id[:4].lower()
    output_path = Path(output_dir) / f"{pdb_code}.pdb"
    
    if not output_path.exists():
        import urllib.request
        url = f"https://files.rcsb.org/download/{pdb_code}.pdb"
        try:
            urllib.request.urlretrieve(url, output_path)
            print(f"Downloaded: {output_path}")
        except Exception as e:
            print(f"Download error: {e}")
    else:
        print(f"Already exists: {output_path}")
    return output_path

if template_a:
    template_a_pdb = download_pdb(template_a)
if template_b:
    template_b_pdb = download_pdb(template_b)

Best template for Chain A: 1NKP_A
Best template for Chain B: 1AN2_A
Already exists: Templates/1nkp.pdb
Already exists: Templates/1an2.pdb


In [15]:
# Create MODELLER alignment and script for Chain A
def create_modeller_files(target_seq, target_id, template_pdb, template_chain, output_dir):
    """
    Create PIR alignment and MODELLER script.
    Automatically detects correct chain ID and residue range from PDB.
    """
    from Bio.PDB import PPBuilder
    import shutil
    
    # Get template sequence and ACTUAL chain info from PDB
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('template', template_pdb)
    ppb = PPBuilder()
    
    # Find protein chains and their residue ranges
    chain_info = {}
    for model in structure:
        for chain in model:
            residues = [r for r in chain.get_residues() if r.id[0] == ' ']  # Standard residues only
            if len(residues) > 10:  # Likely a protein chain
                for pp in ppb.build_peptides(chain):
                    seq = str(pp.get_sequence())
                    if len(seq) > 10:
                        # Get actual residue range
                        res_nums = [r.id[1] for r in residues]
                        chain_info[chain.id] = {
                            'seq': seq,
                            'start': min(res_nums),
                            'end': max(res_nums),
                            'length': len(residues)
                        }
                        break
    
    print(f"  Template PDB chains found:")
    for cid, info in chain_info.items():
        print(f"    Chain {cid}: residues {info['start']}-{info['end']} ({info['length']} aa)")
    
    # Select the best chain (prefer specified, or longest)
    actual_chain = None
    template_seq = ""
    start_res = 1
    end_res = 1
    
    if template_chain in chain_info:
        actual_chain = template_chain
    elif chain_info:
        # Pick the longest protein chain
        actual_chain = max(chain_info.keys(), key=lambda c: chain_info[c]['length'])
        print(f"  Specified chain '{template_chain}' not found, using chain '{actual_chain}'")
    
    if actual_chain:
        template_seq = chain_info[actual_chain]['seq']
        start_res = chain_info[actual_chain]['start']
        end_res = chain_info[actual_chain]['end']
    else:
        raise ValueError(f"No suitable protein chain found in {template_pdb}")
    
    template_code = Path(template_pdb).stem.lower()
    
    # Copy template PDB to MODELLER directory
    template_dest = Path(output_dir) / f"{template_code}.pdb"
    if not template_dest.exists() or True:  # Always copy to ensure latest
        shutil.copy(template_pdb, template_dest)
        print(f"  Copied template to: {template_dest}")
    
    # Create PIR alignment with CORRECT chain ID and residue range
    alignment_filename = f"{target_id}_alignment.pir"
    
    # PIR format: structureX:code:start_res:chain:end_res:chain::::
    pir_content = f""">P1;{template_code}
structureX:{template_code}:{start_res}:{actual_chain}:{end_res}:{actual_chain}::::
{template_seq}*

>P1;{target_id}
sequence:{target_id}::::::::
{target_seq}*
"""
    
    pir_file = Path(output_dir) / alignment_filename
    with open(pir_file, 'w') as f:
        f.write(pir_content)
    
    print(f"  PIR alignment: chain {actual_chain}, residues {start_res}-{end_res}")
    
    # Create MODELLER script
    script_content = f"""# MODELLER script for {target_id}
# Run this script from the {output_dir} directory
from modeller import *
from modeller.automodel import *

log.verbose()
env = Environ()

# Template PDB should be in current directory
env.io.atom_files_directory = ['.']

a = AutoModel(env,
              alnfile='{alignment_filename}',
              knowns='{template_code}',
              sequence='{target_id}',
              assess_methods=(assess.DOPE, assess.GA341))

a.starting_model = 1
a.ending_model = 5  # Generate 5 models

a.make()
"""
    
    script_file = Path(output_dir) / f"model_{target_id}.py"
    with open(script_file, 'w') as f:
        f.write(script_content)
    
    return pir_file, script_file, template_code

# Create files for both chains
print("=== Creating MODELLER Files ===\n")

if template_a:
    print(f"Chain A: template = {template_a}")
    pir_a, script_a, code_a = create_modeller_files(
        seq_a, "ChainA", template_a_pdb, 
        template_a.split('_')[1] if '_' in template_a else 'A',
        MODELLER_DIR
    )
    print(f"  Alignment file: {pir_a.name}")
    print(f"  Script file: {script_a.name}")
    print()

if template_b:
    print(f"Chain B: template = {template_b}")
    pir_b, script_b, code_b = create_modeller_files(
        seq_b, "ChainB", template_b_pdb,
        template_b.split('_')[1] if '_' in template_b else 'A',
        MODELLER_DIR
    )
    print(f"  Alignment file: {pir_b.name}")
    print(f"  Script file: {script_b.name}")

=== Creating MODELLER Files ===

Chain A: template = 1NKP_A
  Template PDB chains found:
    Chain A: residues 897-984 (88 aa)
    Chain B: residues 202-284 (83 aa)
    Chain D: residues 499-581 (83 aa)
    Chain E: residues 704-784 (81 aa)
  Copied template to: Modeller_Templates/1nkp.pdb
  PIR alignment: chain A, residues 897-984
  Alignment file: ChainA_alignment.pir
  Script file: model_ChainA.py

Chain B: template = 1AN2_A
  Template PDB chains found:
    Chain A: residues 22-107 (86 aa)
  Copied template to: Modeller_Templates/1an2.pdb
  PIR alignment: chain A, residues 22-107
  Alignment file: ChainB_alignment.pir
  Script file: model_ChainB.py


In [16]:
# MODELLER execution commands
print("="*60)
print("MODELLER EXECUTION COMMANDS")
print("="*60)
print(f"""
### Step 1: Verify files are in place

Check that these files exist in {MODELLER_DIR}/:
  - ChainA_alignment.pir  (alignment file)
  - ChainB_alignment.pir  (alignment file)
  - model_ChainA.py       (MODELLER script)
  - model_ChainB.py       (MODELLER script)
  - <template>.pdb        (template structure - auto-copied)

### Step 2: Run MODELLER

```bash
# Activate conda environment with MODELLER
conda activate AlphaBald

# Navigate to MODELLER directory (IMPORTANT!)
cd {MODELLER_DIR}

# Run MODELLER for Chain A
python model_ChainA.py

# Run MODELLER for Chain B  
python model_ChainB.py
```

### Step 3: Check output

Best models will be created:
  - ChainA.B99990001.pdb (model 1, check DOPE score in log)
  - ChainA.B99990002.pdb ... ChainA.B99990005.pdb
  - ChainB.B99990001.pdb (model 1)
  - ChainB.B99990002.pdb ... ChainB.B99990005.pdb

### Troubleshooting

If you get "file not found" errors:
1. Make sure you're running from {MODELLER_DIR}/ directory
2. Check template PDB was copied (should be automatic)
3. Verify alignment file exists

If sequence mismatch errors:
1. Check PIR alignment file format
2. Ensure template chain ID is correct
""")

MODELLER EXECUTION COMMANDS

### Step 1: Verify files are in place

Check that these files exist in Modeller_Templates/:
  - ChainA_alignment.pir  (alignment file)
  - ChainB_alignment.pir  (alignment file)
  - model_ChainA.py       (MODELLER script)
  - model_ChainB.py       (MODELLER script)
  - <template>.pdb        (template structure - auto-copied)

### Step 2: Run MODELLER

```bash
# Activate conda environment with MODELLER
conda activate AlphaBald

# Navigate to MODELLER directory (IMPORTANT!)
cd Modeller_Templates

# Run MODELLER for Chain A
python model_ChainA.py

# Run MODELLER for Chain B  
python model_ChainB.py
```

### Step 3: Check output

Best models will be created:
  - ChainA.B99990001.pdb (model 1, check DOPE score in log)
  - ChainA.B99990002.pdb ... ChainA.B99990005.pdb
  - ChainB.B99990001.pdb (model 1)
  - ChainB.B99990002.pdb ... ChainB.B99990005.pdb

### Troubleshooting

If you get "file not found" errors:
1. Make sure you're running from Modeller_Templates/ di

In [17]:
# DSSP output files
dssp_1_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}d1.dssp"
dssp_2_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}d2.dssp"

# Import mkdssp for Python-based DSSP
from mkdssp import dssp

def clean_pdb_for_dssp(input_pdb, output_pdb=None):
    """
    Clean a PDB file for DSSP by removing non-standard records.
    MODELLER adds extra headers that mkdssp can't parse.
    """
    if output_pdb is None:
        output_pdb = str(input_pdb).replace('.pdb', '_clean.pdb')
    
    valid_records = ['ATOM', 'HETATM', 'TER', 'END', 'MODEL', 'ENDMDL']
    
    with open(input_pdb) as f:
        lines = f.readlines()
    
    # Filter to only valid PDB records
    clean_lines = []
    for line in lines:
        record_type = line[:6].strip()
        if record_type in valid_records:
            clean_lines.append(line)
    
    # Make sure file ends with END
    if clean_lines and not clean_lines[-1].startswith('END'):
        clean_lines.append('END\n')
    
    with open(output_pdb, 'w') as f:
        f.writelines(clean_lines)
    
    return output_pdb

def run_dssp_python(input_pdb, output_dssp):
    """
    Run DSSP using Python mkdssp library.
    Automatically cleans PDB file first.
    """
    input_path = Path(input_pdb)
    output_path = Path(output_dssp)
    
    if not input_path.exists():
        print(f"[!] PDB file not found: {input_pdb}")
        print(f"    Run MODELLER first to generate the model.")
        return None, ""
    
    # Check if file is empty
    if input_path.stat().st_size == 0:
        print(f"[!] PDB file is empty: {input_pdb}")
        return None, ""
    
    try:
        # Clean the PDB file first (removes MODELLER headers)
        clean_pdb = clean_pdb_for_dssp(input_path)
        print(f"  Cleaned PDB: {clean_pdb}")
        
        # Run DSSP on cleaned file
        dssp_result = dssp(clean_pdb)
        
        # Extract secondary structure
        ss_string = ""
        result_str = str(dssp_result)
        
        # Save to output file
        with open(output_path, 'w') as f:
            f.write(result_str)
        
        # Parse SS from DSSP output format
        in_residue_section = False
        for line in result_str.split('\n'):
            if '  #  RESIDUE' in line:
                in_residue_section = True
                continue
            if in_residue_section and len(line) > 16:
                aa = line[13:14]
                if aa.isalpha() and aa != '!':  # Valid amino acid, not chain break
                    ss = line[16:17]
                    ss_string += ss if ss != ' ' else 'C'
        
        print(f"[OK] DSSP completed: {output_path}")
        return dssp_result, ss_string
        
    except Exception as e:
        print(f"[ERROR] DSSP failed: {e}")
        print(f"    Make sure the PDB has full backbone atoms (N, CA, C, O)")
        return None, ""

print("="*60)
print("DSSP ANALYSIS")
print("="*60)
print(f"""
DSSP will be run on MODELLER output files:
  - Chain A: {MODELLER_DIR}/ChainA.B99990001.pdb -> {dssp_1_output}
  - Chain B: {MODELLER_DIR}/ChainB.B99990001.pdb -> {dssp_2_output}

Prerequisites:
  1. MODELLER must have completed successfully
  2. Model files must exist in {MODELLER_DIR}/
""")

DSSP ANALYSIS

DSSP will be run on MODELLER output files:
  - Chain A: Modeller_Templates/ChainA.B99990001.pdb -> Problem_1_outputs/p38d1.dssp
  - Chain B: Modeller_Templates/ChainB.B99990001.pdb -> Problem_1_outputs/p38d2.dssp

Prerequisites:
  1. MODELLER must have completed successfully
  2. Model files must exist in Modeller_Templates/



In [21]:
# Run DSSP on MODELLER models
print("=== Running DSSP Analysis ===\n")

# Chain A
model_a = Path(MODELLER_DIR) / "ChainA.B99990001.pdb"
print(model_a)
if model_a.exists():
    run_dssp_python(model_a, dssp_1_output)
    ss_a, residues_a = clean_pdb_for_dssp(dssp_1_output)
    if ss_a:
        print(f"  Chain A SS: {ss_a[:50]}..." if len(ss_a) > 50 else f"  Chain A SS: {ss_a}")
        print(f"  Length: {len(ss_a)} residues")
else:
    print(f"[!] Chain A model not found: {model_a}")
    print(f"    Run MODELLER first: cd {MODELLER_DIR} && python model_ChainA.py")

print()

# Chain B
model_b = Path(MODELLER_DIR) / "ChainB.B99990001.pdb"
if model_b.exists():
    run_dssp_python(model_b, dssp_2_output)
    ss_b, residues_b = parse_dssp_output(dssp_2_output)
    if ss_b:
        print(f"  Chain B SS: {ss_b[:50]}..." if len(ss_b) > 50 else f"  Chain B SS: {ss_b}")
        print(f"  Length: {len(ss_b)} residues")
else:
    print(f"[!] Chain B model not found: {model_b}")
    print(f"    Run MODELLER first: cd {MODELLER_DIR} && python model_ChainB.py")

print("""
=== DSSP Code Legend ===
H = α-helix       G = 3₁₀-helix    I = π-helix
E = β-strand      B = β-bridge
T = Turn          S = Bend         C/  = Coil/loop
""")

=== Running DSSP Analysis ===

Modeller_Templates/ChainA.B99990001.pdb
  Cleaned PDB: Modeller_Templates/ChainA.B99990001_clean.pdb
[ERROR] DSSP failed: Empty file?
    Make sure the PDB has full backbone atoms (N, CA, C, O)


Dropped unsupported records: Modell
Error parsing PDB


FileNotFoundError: [Errno 2] No such file or directory: 'Problem_1_outputs/p38d1.dssp'

In [30]:
# Run DSSP on MODELLER models
print("=== Running DSSP Analysis ===\n")

# Chain A
model_a = Path(MODELLER_DIR) / "ChainA.B99990001.pdb"
ss_a = ""
print(f"Chain A model: {model_a}")
if model_a.exists():
    _, ss_a = run_dssp_python(model_a, dssp_1_output)
    if ss_a:
        print(f"  SS: {ss_a[:60]}..." if len(ss_a) > 60 else f"  SS: {ss_a}")
        print(f"  Length: {len(ss_a)} residues")
else:
    print(f"[!] Not found - run MODELLER first:")
    print(f"    cd {MODELLER_DIR} && python model_ChainA.py")

print()

# Chain B
model_b = Path(MODELLER_DIR) / "ChainB.B99990001.pdb"
ss_b = ""
print(f"Chain B model: {model_b}")
if model_b.exists():
    _, ss_b = run_dssp_python(model_b, dssp_2_output)
    if ss_b:
        print(f"  SS: {ss_b[:60]}..." if len(ss_b) > 60 else f"  SS: {ss_b}")
        print(f"  Length: {len(ss_b)} residues")
else:
    print(f"[!] Not found - run MODELLER first:")
    print(f"    cd {MODELLER_DIR} && python model_ChainB.py")

print("""
=== DSSP Code Legend ===
H = α-helix       G = 3₁₀-helix    I = π-helix
E = β-strand      B = β-bridge
T = Turn          S = Bend         C/  = Coil/loop
""")

=== Running DSSP Analysis ===

Chain A model: Modeller_Templates/ChainA.B99990001.pdb
  Cleaned PDB: Modeller_Templates/ChainA.B99990001_clean.pdb
[ERROR] DSSP failed: Empty file?
    Make sure the PDB has full backbone atoms (N, CA, C, O)

Chain B model: Modeller_Templates/ChainB.B99990001.pdb
  Cleaned PDB: Modeller_Templates/ChainB.B99990001_clean.pdb
[ERROR] DSSP failed: Empty file?
    Make sure the PDB has full backbone atoms (N, CA, C, O)

=== DSSP Code Legend ===
H = α-helix       G = 3₁₀-helix    I = π-helix
E = β-strand      B = β-bridge
T = Turn          S = Bend         C/  = Coil/loop



Dropped unsupported records: Modell
Error parsing PDB
Dropped unsupported records: Modell
Error parsing PDB


In [19]:
# Save sequences for web submission
print("=== Sequences for SS Prediction Web Servers ===")
print(f"\nSubmit these sequences to PSIPRED or JPred:")
print(f"\n>ChainA\n{seq_a}")
print(f"\n>ChainB\n{seq_b}")

# Create alignment files comparing predicted vs actual secondary structure
aln_1_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}e1.aln"
aln_2_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}e2.aln"

print(f"""
=== Creating SS Comparison Alignment ===

After getting results from:
1. DSSP (from step d)
2. PSIPRED or JPred prediction

Create alignment files showing:
- Sequence
- DSSP secondary structure
- Predicted secondary structure
- Matches (*) vs mismatches

Save as:
- {aln_1_output}
- {aln_2_output}
""")

=== Sequences for SS Prediction Web Servers ===

Submit these sequences to PSIPRED or JPred:

>ChainA
NVKRRTHNVLERQRRNELKRSFFAFRDQIPELENNEKAPKVVILKKFTATATATTAYILSVQAEEQKLISEEDLLRKRREQLKHKLEQL

>ChainB
DALAHHNALELDLADHIKDSFHSLSLFLDSVPSLQGEKASRAQILDKFTEYIQYMRRKNHTHQQDIDDLKRQNALLEQQVRAL

=== Creating SS Comparison Alignment ===

After getting results from:
1. DSSP (from step d)
2. PSIPRED or JPred prediction

Create alignment files showing:
- Sequence
- DSSP secondary structure
- Predicted secondary structure
- Matches (*) vs mismatches

Save as:
- Problem_1_outputs/p38e1.aln
- Problem_1_outputs/p38e2.aln



In [ ]:
def create_ss_comparison(sequence, dssp_ss, predicted_ss, output_file):
    """
    Create alignment file comparing DSSP and predicted secondary structure.
    
    Parameters:
    - sequence: amino acid sequence
    - dssp_ss: DSSP secondary structure string (H, E, C, etc.)
    - predicted_ss: Predicted SS string from PSIPRED/JPred
    - output_file: output alignment file path
    """
    # Simplify to 3-state (H=helix, E=strand, C=coil)
    def simplify_ss(ss):
        result = ""
        for c in ss:
            if c in ['H', 'G', 'I']:
                result += 'H'
            elif c in ['E', 'B']:
                result += 'E'
            else:
                result += 'C'
        return result
    
    dssp_simple = simplify_ss(dssp_ss)
    pred_simple = simplify_ss(predicted_ss)
    
    # Calculate agreement
    matches = sum(1 for d, p in zip(dssp_simple, pred_simple) if d == p)
    accuracy = matches / len(sequence) * 100 if len(sequence) > 0 else 0
    
    agreement = ''.join(['*' if d == p else ' ' for d, p in zip(dssp_simple, pred_simple)])
    
    content = f""">Sequence
{sequence}
>DSSP_SS (experimental)
{dssp_ss}
>Predicted_SS
{predicted_ss}
>Agreement (* = match, accuracy = {accuracy:.1f}%)
{agreement}
"""
    
    with open(output_file, 'w') as f:
        f.write(content)
    
    print(f"Saved: {output_file}")
    print(f"Agreement: {accuracy:.1f}%")
    return accuracy

# Example usage (fill in actual SS strings after running DSSP and prediction)
# create_ss_comparison(seq_a, "CCCHHHHHHHHHHCCCCEEEEECCC...", "CCHHHHHHHHHHHCCCEEEEEECCC...", aln_1_output)

---
## f) Structure Validation with ProSA (0.5 pts)

### Biological Background: ProSA Energy Analysis

**What ProSA measures:**
- Statistical potential energy based on known protein structures
- **Z-score**: Overall model quality
  - Native proteins: Z-score typically -4 to -12
  - Poor models: Z-score > -4 or positive
- **Local energy profile**: Identifies problematic regions

**Interpreting ProSA results:**
- Blue region: Good Z-score range
- Outside blue: Unusual, possibly problematic
- High local energy peaks: Structural errors

**ProSA Web Server:**
https://prosa.services.came.sbg.ac.at/prosa.php

In [20]:
energy_plot_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}f.png"

print(f"""
=== ProSA Validation Instructions ===

1. Go to ProSA web server:
   https://prosa.services.came.sbg.ac.at/prosa.php

2. Upload your MODELLER model:
   - For full heterodimer: combine both chains into one PDB
   - Or analyze each chain separately

3. Submit and wait for results

4. Download/screenshot the energy profile image

5. Save as: {energy_plot_output}

=== What to Report ===

- Z-score value
- Is it within expected range for this protein size?
- Any high-energy regions visible in the profile?
- Compare with template structure Z-score if available
""")


=== ProSA Validation Instructions ===

1. Go to ProSA web server:
   https://prosa.services.came.sbg.ac.at/prosa.php

2. Upload your MODELLER model:
   - For full heterodimer: combine both chains into one PDB
   - Or analyze each chain separately

3. Submit and wait for results

4. Download/screenshot the energy profile image

5. Save as: Problem_1_outputs/p38f.png

=== What to Report ===

- Z-score value
- Is it within expected range for this protein size?
- Any high-energy regions visible in the profile?
- Compare with template structure Z-score if available



---
## g) Identify Structural Problems (0.5 pts)

### Biological Background: B-factors and Structural Problems

**What are B-factors (Temperature factors)?**
- Measure atomic displacement/flexibility
- High B-factors indicate:
  - Flexible regions
  - Poorly resolved in X-ray
  - Potential modeling errors

**Common structural problems:**
- Loops with poor density
- Terminal regions
- Surface residues far from core
- Regions without good template coverage

In [2]:
problems_image_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}g.png"

# Analyze B-factors from original structure
def analyze_bfactors(atoms, chain_name):
    """Analyze B-factors to identify problematic regions."""
    bfactors = [(a['resnum'], a['aa'], a['bfactor']) for a in atoms]
    df = pd.DataFrame(bfactors, columns=['ResNum', 'AA', 'B-factor'])
    
    # High B-factors indicate flexible/problematic regions
    mean_bf = df['B-factor'].mean()
    std_bf = df['B-factor'].std()
    threshold = mean_bf + 2 * std_bf
    
    problematic = df[df['B-factor'] > threshold]
    
    print(f"\n=== {chain_name} B-factor Analysis ===")
    print(f"Mean B-factor: {mean_bf:.2f}")
    print(f"Std: {std_bf:.2f}")
    print(f"Threshold (mean + 2*std): {threshold:.2f}")
    print(f"\nProblematic residues (B-factor > threshold):")
    
    return df, problematic

bf_a, prob_a = analyze_bfactors(chain_a_atoms, "Chain A")
display(prob_a)

bf_b, prob_b = analyze_bfactors(chain_b_atoms, "Chain B")
display(prob_b)

NameError: name 'Path' is not defined

In [ ]:
# Visualize B-factors
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Chain A
axes[0].bar(bf_a['ResNum'], bf_a['B-factor'], color='steelblue', alpha=0.7)
threshold_a = bf_a['B-factor'].mean() + 2*bf_a['B-factor'].std()
axes[0].axhline(y=threshold_a, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold_a:.1f})')
axes[0].set_xlabel('Residue Number', fontsize=12)
axes[0].set_ylabel('B-factor', fontsize=12)
axes[0].set_title('Chain A B-factor Profile - High values indicate problematic regions', fontsize=14)
axes[0].legend()

# Highlight problematic residues
for _, row in prob_a.iterrows():
    axes[0].bar(row['ResNum'], row['B-factor'], color='red', alpha=0.8)

# Chain B
axes[1].bar(bf_b['ResNum'], bf_b['B-factor'], color='darkorange', alpha=0.7)
threshold_b = bf_b['B-factor'].mean() + 2*bf_b['B-factor'].std()
axes[1].axhline(y=threshold_b, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold_b:.1f})')
axes[1].set_xlabel('Residue Number', fontsize=12)
axes[1].set_ylabel('B-factor', fontsize=12)
axes[1].set_title('Chain B B-factor Profile - High values indicate problematic regions', fontsize=14)
axes[1].legend()

for _, row in prob_b.iterrows():
    axes[1].bar(row['ResNum'], row['B-factor'], color='red', alpha=0.8)

plt.tight_layout()
plt.savefig(problems_image_output, dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: {problems_image_output}")

In [ ]:
# Summary of problematic regions
problems_answer = f"""
=== ANSWER g) Structural Problems ===

CHAIN A PROBLEMATIC RESIDUES:
Positions: {list(prob_a['ResNum'].values)}
Residues: {list(prob_a['AA'].values)}

CHAIN B PROBLEMATIC RESIDUES:
Positions: {list(prob_b['ResNum'].values)}
Residues: {list(prob_b['AA'].values)}

INTERPRETATION:
- These residues have unusually high B-factors
- Likely represent flexible loops or poorly modeled regions
- Should be targeted for loop refinement in step k)

ADDITIONAL VALIDATION:
- Check Ramachandran plot (MolProbity)
- Check for steric clashes
- Compare with ProSA local energy profile
"""

print(problems_answer)

---
## h) Dimer Formation Assessment (0.5 pts)

### Biological Background: Protein-Protein Interfaces

**What makes a good dimer interface?**
- Hydrophobic core at interface
- Complementary shape (knob-into-hole packing)
- Hydrogen bonds and salt bridges
- Buried surface area typically 1000-3000 Å²

**Interface analysis methods:**
- Distance-based: residues within 5-8 Å of partner
- Accessible surface area: residues that become buried

**For heterodimers:**
- Often mediated by specific domains (e.g., leucine zippers, HLH)
- Conserved interface residues across family members

In [ ]:
dimer_image_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}h.png"

def calculate_interface(atoms_a, atoms_b, distance_cutoff=8.0):
    """Calculate interface residues between two chains (CA-CA distance)."""
    interface_a = set()
    interface_b = set()
    contacts = []
    
    for a in atoms_a:
        for b in atoms_b:
            dist = np.sqrt((a['x']-b['x'])**2 + (a['y']-b['y'])**2 + (a['z']-b['z'])**2)
            if dist < distance_cutoff:
                interface_a.add((a['resnum'], a['aa']))
                interface_b.add((b['resnum'], b['aa']))
                contacts.append({
                    'ChainA_Res': f"{a['aa']}{a['resnum']}",
                    'ChainB_Res': f"{b['aa']}{b['resnum']}",
                    'Distance': round(dist, 2)
                })
    
    return sorted(interface_a), sorted(interface_b), pd.DataFrame(contacts)

interface_a, interface_b, contacts_df = calculate_interface(chain_a_atoms, chain_b_atoms)

print(f"=== Dimer Interface Analysis ===")
print(f"\nChain A interface residues ({len(interface_a)}):") 
print([f"{aa}{num}" for num, aa in interface_a])

print(f"\nChain B interface residues ({len(interface_b)}):")
print([f"{aa}{num}" for num, aa in interface_b])

# Show closest contacts
print(f"\nClosest inter-chain contacts:")
display(contacts_df.drop_duplicates().sort_values('Distance').head(15))

In [ ]:
# Assess dimer viability
dimer_answer = f"""
=== ANSWER h) Dimer Formation Assessment ===

INTERFACE STATISTICS:
- Chain A interface residues: {len(interface_a)}
- Chain B interface residues: {len(interface_b)}
- Total interface contacts: {len(contacts_df)}

ASSESSMENT:
"""

if len(interface_a) > 10 and len(interface_b) > 10:
    dimer_answer += """The structure DOES show a viable dimer interface:
- Sufficient number of interface residues on both chains
- Multiple close contacts observed
- Interface likely stabilizes heterodimer formation
"""
else:
    dimer_answer += """The interface appears WEAK or incomplete:
- Few interface residues detected
- May need structural refinement
- Check if chains are properly positioned
"""

dimer_answer += f"""
INTERFACE CHARACTER:
- Look for hydrophobic residues (L, I, V, F, W) at interface
- Look for charged residues (R, K, E, D) for salt bridges
- The interface should be complementary in shape
"""

print(dimer_answer)

---
## i) Functional Residue Conservation (0.5 pts)

### Biological Background: Conservation and Function

**Why conserved residues matter:**
- Evolution preserves functionally important residues
- Highly conserved = likely essential for function
- Variable residues often surface-exposed, not critical

**How to identify conserved residues:**
1. Create multiple sequence alignment (MSA) with homologs
2. Calculate conservation at each position
3. Positions with >80% conservation are likely functional

In [ ]:
# Create MSA and analyze conservation
def create_msa_with_homologs(fasta_file, output_alignment, chain_name, num_homologs=10):
    """Create MSA with top homologs from SwissProt."""
    # Get sequences from SwissProt blast results
    blast_out = Path(TEMP_DIR) / f"msa_blast_{chain_name}.out"
    
    cmd = [
        "psiblast", "-query", str(fasta_file), "-db", SWISSPROT_DB,
        "-out", str(blast_out), "-outfmt", "6 sacc",
        "-max_target_seqs", str(num_homologs), "-evalue", "1e-10"
    ]
    subprocess.run(cmd, capture_output=True)
    
    if blast_out.exists() and blast_out.stat().st_size > 0:
        accs = blast_out.read_text().strip().split('\n')[:num_homologs]
        if accs and accs[0]:
            acc_list = ','.join([a for a in accs if a])
            seqs_file = Path(TEMP_DIR) / f"homolog_seqs_{chain_name}.fa"
            
            # Extract sequences from database
            cmd = ["blastdbcmd", "-db", SWISSPROT_DB, "-entry", acc_list,
                   "-out", str(seqs_file)]
            subprocess.run(cmd, capture_output=True)
            
            # Combine target with homologs
            combined = Path(TEMP_DIR) / f"combined_seqs_{chain_name}.fa"
            with open(combined, 'w') as out:
                out.write(Path(fasta_file).read_text())
                if seqs_file.exists():
                    out.write(seqs_file.read_text())
            
            # Run ClustalW
            cmd = ["clustalw", f"-INFILE={combined}",
                   f"-OUTFILE={output_alignment}",
                   "-OUTPUT=FASTA", "-OUTORDER=INPUT"]
            result = subprocess.run(cmd, capture_output=True)
            
            if Path(output_alignment).exists():
                print(f"Created MSA: {output_alignment}")
                return True
    
    print(f"Could not create MSA for {chain_name}")
    return False

# Create MSAs
os.makedirs(ALIGNMENTS_DIR, exist_ok=True)
msa_a = Path(ALIGNMENTS_DIR) / "chainA_msa.fa"
msa_b = Path(ALIGNMENTS_DIR) / "chainB_msa.fa"

create_msa_with_homologs(fasta_a, msa_a, "chainA")
create_msa_with_homologs(fasta_b, msa_b, "chainB")

In [ ]:
# Analyze conservation
def analyze_conservation(msa_file, threshold=0.8):
    """Identify highly conserved positions in MSA."""
    if not Path(msa_file).exists():
        print(f"MSA file not found: {msa_file}")
        return []
    
    # Read alignment
    alignment = list(SeqIO.parse(msa_file, 'fasta'))
    if len(alignment) < 2:
        print("Not enough sequences in alignment")
        return []
    
    # Calculate conservation at each position
    aln_length = len(alignment[0].seq)
    num_seqs = len(alignment)
    conserved = []
    
    for pos in range(aln_length):
        column = [str(rec.seq[pos]) for rec in alignment]
        # Count most common residue
        from collections import Counter
        counts = Counter(c for c in column if c != '-')
        if counts:
            most_common, count = counts.most_common(1)[0]
            conservation = count / num_seqs
            if conservation >= threshold:
                conserved.append({
                    'Position': pos + 1,
                    'Residue': most_common,
                    'Conservation': round(conservation * 100, 1)
                })
    
    return pd.DataFrame(conserved)

print("\n=== Chain A Conserved Residues (>80%) ===")
conserved_a = analyze_conservation(msa_a, threshold=0.8)
display(conserved_a)

print("\n=== Chain B Conserved Residues (>80%) ===")
conserved_b = analyze_conservation(msa_b, threshold=0.8)
display(conserved_b)

---
## j) Active Site Visualization (0.5 pts)

### Biological Background: Active/Functional Sites

**For transcription factors (like Myc-Max):**
- The "active site" is the DNA-binding region
- Basic region contacts DNA
- HLH/leucine zipper mediates dimerization

**Visualization tips:**
- Highlight conserved residues
- Show residues involved in function
- Use PyMOL for publication-quality images

In [ ]:
active_site_image = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}j.png"

# PyMOL script for active site visualization
pymol_script = f"""
# PyMOL script for active site visualization
# Run: pymol -c script.pml (or open in PyMOL GUI)

# Load structure
load {pdb_combined}, heterodimer

# Basic setup
bg_color white
set cartoon_fancy_helices, 1
set cartoon_highlight_color, grey50

# Show cartoon representation
hide all
show cartoon, heterodimer

# Color chains differently
color marine, chain A
color orange, chain B

# Highlight conserved/functional residues
# (Fill in actual residue numbers from conservation analysis)
# select functional, resi 10+15+20+25 and chain A
# show sticks, functional
# color red, functional

# Highlight interface
# select interface_a, chain A and resi {'+'.join([str(r[0]) for r in interface_a[:20]])}
# select interface_b, chain B and resi {'+'.join([str(r[0]) for r in interface_b[:20]])}
# show sticks, interface_a or interface_b

# Set view and render
orient
zoom complete=1
ray 1200, 900
png {active_site_image}, dpi=300
"""

script_file = Path(OUTPUT_DIR) / "visualize_active_site.pml"
with open(script_file, 'w') as f:
    f.write(pymol_script)

print(f"Created PyMOL script: {script_file}")
print(f"""
=== Active Site Visualization Instructions ===

1. Open PyMOL
2. Run the script:
   pymol {script_file}
   
   Or in PyMOL GUI: File > Run Script > select the .pml file

3. Modify the script to highlight:
   - Conserved residues (from step i)
   - Interface residues (from step h)
   - Known functional residues from literature

4. Save image as: {active_site_image}
""")

---
## k) Fix Structural Problems (0.5 pts)

### Strategy: Loop Refinement with MODELLER

**What we're fixing:**
- High B-factor regions identified in step g
- ProSA high-energy regions from step f

**MODELLER loop refinement:**
- Focuses on specific loop regions
- Samples many conformations
- Selects best by energy

In [ ]:
fixed_model_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}k.pdb"
fixed_prosa_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}k.png"

# Identify regions to refine (from B-factor analysis)
def get_loop_regions(problematic_df, window=3):
    """Convert problematic residues to loop regions for refinement."""
    if len(problematic_df) == 0:
        return []
    
    positions = sorted(problematic_df['ResNum'].values)
    regions = []
    
    # Expand each position by window and merge overlapping
    for pos in positions:
        start = max(1, pos - window)
        end = pos + window
        
        if regions and start <= regions[-1][1] + 1:
            # Merge with previous region
            regions[-1] = (regions[-1][0], end)
        else:
            regions.append((start, end))
    
    return regions

loop_regions_a = get_loop_regions(prob_a)
loop_regions_b = get_loop_regions(prob_b)

print(f"Regions to refine in Chain A: {loop_regions_a}")
print(f"Regions to refine in Chain B: {loop_regions_b}")

In [ ]:
# Create MODELLER loop refinement script
def create_loop_refinement_script(input_model, target_id, loop_regions, output_dir):
    """
    Create MODELLER script for loop refinement.
    All paths are relative to output_dir.
    """
    
    # Get just the filename (model should already be in output_dir)
    model_filename = Path(input_model).name
    
    # Format loop selection
    loop_selection = ""
    for i, (start, end) in enumerate(loop_regions):
        loop_selection += f"        self.residue_range('{start}:', '{end}:'),\n"
    
    script = f"""# MODELLER Loop Refinement Script for {target_id}
# Run this script from the {output_dir} directory
from modeller import *
from modeller.automodel import *

log.verbose()
env = Environ()

# All files should be in current directory
env.io.atom_files_directory = ['.']

class MyLoop(LoopModel):
    def select_loop_atoms(self):
        return Selection(
{loop_selection}        )

m = MyLoop(env,
           inimodel='{model_filename}',
           sequence='{target_id}')

m.loop.starting_model = 1
m.loop.ending_model = 10  # Generate 10 loop conformations
m.loop.md_level = refine.slow  # Thorough refinement

m.make()
"""
    
    script_file = Path(output_dir) / f"refine_loops_{target_id}.py"
    with open(script_file, 'w') as f:
        f.write(script)
    
    return script_file

# Define loop regions to refine (modify based on B-factor analysis)
# These should be the high B-factor regions identified in step g
def get_loop_regions(bfactor_df, threshold=50.0):
    """Extract loop regions with high B-factors."""
    if bfactor_df is None or len(bfactor_df) == 0:
        return []
    
    high_bf = bfactor_df[bfactor_df['B-factor'] > threshold]
    if len(high_bf) == 0:
        return []
    
    # Group consecutive residues into regions
    regions = []
    residues = sorted(high_bf['Residue'].tolist())
    
    if not residues:
        return []
    
    start = residues[0]
    end = residues[0]
    
    for res in residues[1:]:
        if res <= end + 2:  # Allow small gaps
            end = res
        else:
            if end - start >= 2:  # Only include regions of 3+ residues
                regions.append((start, end))
            start = res
            end = res
    
    if end - start >= 2:
        regions.append((start, end))
    
    return regions

# Get loop regions from B-factor analysis (if available)
try:
    loop_regions_a = get_loop_regions(bfactor_a_df) if 'bfactor_a_df' in dir() else []
    loop_regions_b = get_loop_regions(bfactor_b_df) if 'bfactor_b_df' in dir() else []
except:
    loop_regions_a = []
    loop_regions_b = []

print("=== Loop Regions to Refine ===")
print(f"Chain A: {loop_regions_a if loop_regions_a else 'None identified (run B-factor analysis first)'}")
print(f"Chain B: {loop_regions_b if loop_regions_b else 'None identified (run B-factor analysis first)'}")

# Create refinement scripts
if loop_regions_a:
    refine_script_a = create_loop_refinement_script(
        f"ChainA.B99990001.pdb",  # Just filename, not full path
        "ChainA",
        loop_regions_a,
        MODELLER_DIR
    )
    print(f"\nCreated: {refine_script_a}")

if loop_regions_b:
    refine_script_b = create_loop_refinement_script(
        f"ChainB.B99990001.pdb",  # Just filename, not full path
        "ChainB", 
        loop_regions_b,
        MODELLER_DIR
    )
    print(f"Created: {refine_script_b}")

In [ ]:
fixed_model_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}k.pdb"
fixed_prosa_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}k_prosa.png"

print("="*60)
print("LOOP REFINEMENT EXECUTION COMMANDS")
print("="*60)
print(f"""
### Prerequisites
- MODELLER initial models must exist (ChainA.B99990001.pdb, ChainB.B99990001.pdb)
- Run initial MODELLER first if not done

### Step 1: Run loop refinement

```bash
# Activate environment
conda activate AlphaBald

# Navigate to MODELLER directory (IMPORTANT!)
cd {MODELLER_DIR}

# Refine Chain A loops (if script was created)
python refine_loops_ChainA.py

# Refine Chain B loops (if script was created)
python refine_loops_ChainB.py
```

### Step 2: Check output

Refined models will be named:
  - ChainA.BL00010001.pdb through ChainA.BL00010010.pdb
  - ChainB.BL00010001.pdb through ChainB.BL00010010.pdb

Select models with lowest DOPE score from the log file.

### Step 3: Create final model

```bash
# Combine refined chains (if modeling heterodimer)
# Or just copy the best single-chain model

# Copy best model to output
cp ChainA.BL00010001.pdb ../{OUTPUT_DIR}/{OUTPUT_PREFIX}k.pdb
```

### Step 4: Re-validate with ProSA

1. Go to: https://prosa.services.came.sbg.ac.at/prosa.php
2. Upload refined model: {fixed_model_output}
3. Compare Z-score with original model
4. Save energy plot as: {fixed_prosa_output}
""")

---
## l) New Model Dimer Assessment (0.5 pts)

After fixing the structural problems, reassess the dimer.

In [ ]:
dimer_model_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}l.pdb"
dimer_comparison_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}l.png"

print(f"""
=== Final Dimer Model Assessment ===

After fixing structural problems:

1. COMBINE REFINED CHAINS:
   - Take refined Chain A and Chain B
   - Superimpose onto original positions
   - Combine into single PDB file
   - Save as: {dimer_model_output}

2. VALIDATE DIMER:
   - Re-run ProSA on combined structure
   - Check Z-score improvement
   - Verify interface is maintained

3. CREATE COMPARISON FIGURE:
   - Before/after comparison
   - Show Z-score improvement
   - Highlight fixed regions
   - Save as: {dimer_comparison_output}

4. PYMOL COMMANDS FOR COMPARISON:

   # Load both models
   load original_model.pdb, original
   load {dimer_model_output}, refined
   
   # Superimpose
   align refined, original
   
   # Color by B-factor or by chain
   spectrum b, blue_white_red, original
   color green, refined
   
   # Save image
   ray 1200, 900
   png {dimer_comparison_output}, dpi=300
""")

---
## Output Files Checklist

In [ ]:
expected_outputs = [
    (f"{OUTPUT_PREFIX}b1.hmm", "HMM profile for protein 1 (Pfam family)"),
    (f"{OUTPUT_PREFIX}b2.hmm", "HMM profile for protein 2 (Pfam family)"),
    (f"{OUTPUT_PREFIX}d1.dssp", "DSSP secondary structure for chain 1"),
    (f"{OUTPUT_PREFIX}d2.dssp", "DSSP secondary structure for chain 2"),
    (f"{OUTPUT_PREFIX}e1.aln", "SS prediction vs DSSP alignment for chain 1"),
    (f"{OUTPUT_PREFIX}e2.aln", "SS prediction vs DSSP alignment for chain 2"),
    (f"{OUTPUT_PREFIX}f.png", "ProSA energy profile plot"),
    (f"{OUTPUT_PREFIX}g.png", "Structural problems visualization (B-factors)"),
    (f"{OUTPUT_PREFIX}h.png", "Dimer interface visualization"),
    (f"{OUTPUT_PREFIX}j.png", "Active site/functional residues visualization"),
    (f"{OUTPUT_PREFIX}k.pdb", "Corrected/refined structure"),
    (f"{OUTPUT_PREFIX}k.png", "ProSA comparison before/after"),
    (f"{OUTPUT_PREFIX}l.pdb", "Final dimer model"),
    (f"{OUTPUT_PREFIX}l.png", "Dimer comparison image"),
]

print(f"=== Output Files Checklist ({OUTPUT_DIR}/) ===")
print("\nCheck each file as you complete it:\n")
for filename, description in expected_outputs:
    filepath = Path(OUTPUT_DIR) / filename
    status = "[x]" if filepath.exists() else "[ ]"
    print(f"{status} {filename:20s} - {description}")